In [ ]:
#Code taken from Clarifai API model of using facial recognition to identify ethnicity


#Limits: 
# 1. 1000 calls per month
# 2. 128 images in one API call
# 3. image input must be less than 20MB
# 4. Clear images, no less than 50x50

In [3]:
#pip install clarifai-grpc

  Obtaining dependency information for clarifai-grpc from https://files.pythonhosted.org/packages/31/e2/bc10436e525584c42910d16a558dd5a1378ddf89ceb5561097c484b9d0cb/clarifai_grpc-10.0.2-py3-none-any.whl.metadata
  Obtaining dependency information for grpcio>=1.44.0 from https://files.pythonhosted.org/packages/4a/78/a8d3314e75e73fe025d39a2fa336c415dc12ce4e4166de756f9e3579111c/grpcio-1.60.0-cp39-cp39-win_amd64.whl.metadata
   ---------------------------------------- 224.8/224.8 kB 2.8 MB/s eta 0:00:00
   ---------------------------------------- 3.7/3.7 MB 5.2 MB/s eta 0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.42.0
    Uninstalling grpcio-1.42.0:
      Successfully uninstalled grpcio-1.42.0
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import csv
csv_file = open("Images_URL.csv", "r")

#Create list with image sources
IMAGE_URL = [item for sublist in csv.reader(csv_file, delimiter=",") for item in sublist]

#Remove undefined urls
remove_words = "undefined"
IMAGE_URL = [word for word in IMAGE_URL if word != remove_words]
print(IMAGE_URL)

for x in IMAGE_URL:
    print(x)

In [ ]:
#Use ethnicity model example and apply to DNNL entrepreneurs

# Specify PAT token
PAT = '79fb7778a82443479813229c210dcede'
# Specify user_id and app_id 
USER_ID = '2tbrpyd5q8bu'
APP_ID = 'ethnicity-detection'
# Specify model and and image url list
MODEL_ID = 'ethnicity-demographics-recognition'
MODEL_VERSION_ID = 'b2897edbda314615856039fb0c489796'
BATCH_SIZE = 11

#Import all important features from Clarifai
from clarifai_grpc.channel.clarifai_channel import ClarifaiChannel
from clarifai_grpc.grpc.api import resources_pb2, service_pb2, service_pb2_grpc
from clarifai_grpc.grpc.api.status import status_code_pb2

channel = ClarifaiChannel.get_grpc_channel()
stub = service_pb2_grpc.V2Stub(channel)

metadata = (('authorization', 'Key ' + PAT),)

#Apply function to chunck all data into smaller list - better for handling the model
def chunk_list(input_list, batch_size):
    return [input_list[i:i + batch_size] for i in range(0, len(input_list), batch_size)]

url_batches = chunk_list(IMAGE_URL, BATCH_SIZE)
output_concepts = []

# Iterate through batches
for batch in url_batches:
    inputs=[
        resources_pb2.Input(
            data=resources_pb2.Data(
                image=resources_pb2.Image(
                    url=url
                )
            )
        ) for url in batch
        
    ]
    userDataObject = resources_pb2.UserAppIDSet(user_id=USER_ID, app_id=APP_ID)
    post_model_outputs_response = stub.PostModelOutputs(
        service_pb2.PostModelOutputsRequest(
            user_app_id=userDataObject,  
            model_id=MODEL_ID,
            version_id=MODEL_VERSION_ID,
            inputs=inputs
        ), 
        metadata=metadata
    )

    if post_model_outputs_response.status.code != status_code_pb2.SUCCESS:
        print(post_model_outputs_response.status)
        raise Exception("Post model outputs failed, status: " + post_model_outputs_response.status.description)

#Get output
    outputs = post_model_outputs_response.outputs

# Iterate through outputs and save predictions
    for x in outputs:
        concepts = []
        for y in x.data.concepts:
            concepts.append((y.name, y.value))
        output_concepts.append(concepts)

# Print predicted ethnicities
    for idx, concepts in enumerate(output_concepts):
        print(f"Predicted concepts for output {idx + 1}:")
        for concept_name, concept_value in concepts:
            print(f"{concept_name} {concept_value:.2f}")
        print("\n")


In [ ]:
#Extract the first ethnicity with the highest probability
import csv
csv_name = 'Ethnicity_First.csv'
data=[]
for lists in output_concepts:
    data.append(lists[0])

print(data)


# Save in CSV file
with open(csv_name, 'w', newline='') as csv_file:
    csv_writer = csv.writer(csv_file)
    for x in data:
        csv_writer.writerow([x])

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

#Create dataframe with ethnicity and probability columns
columns= ['Ethnicity', 'Probability']
data= pd.read_csv("Ethnicity_First.csv", names=columns )
data[['Ethnicity', 'Probability']] = data['Ethnicity'].str.extract(r"\('([^']+)', ([^)]+)\)")

# Display the result
display(data)

#Select ethnicity column
ethnicity=data['Ethnicity']

#Count values based on ethnicity
black=data.value_counts(subset=ethnicity)['Black']
print(black)

east_asian=data.value_counts(subset=ethnicity)['East Asian']
print(east_asian)

white=data.value_counts(subset=ethnicity)['White']
print(white)

southeast_asian=data.value_counts(subset=ethnicity)['Southeast Asian']
print(southeast_asian)

latino_hispanic=data.value_counts(subset=ethnicity)['Latino_Hispanic']
print(latino_hispanic)

indian=data.value_counts(subset=ethnicity)['Indian']
print(indian)

middle_eastern=data.value_counts(subset=ethnicity)['Middle Eastern']
print(middle_eastern)

#Count total number of values
total= len(data)

#Get percentages of all ethnicities
fraction_black= ((black)/(total))*100
percentage_black= round(fraction_black)

fraction_white= ((white)/(total))*100
percentage_white= round(fraction_white)

fraction_indian= ((indian)/(total))*100
percentage_indian= round(fraction_indian)

fraction_sa= ((southeast_asian)/(total))*100
percentage_sa= round(fraction_sa)

fraction_ea= ((east_asian)/(total))*100
percentage_ea= round(fraction_ea)

fraction_me= ((middle_eastern)/(total))*100
percentage_me= round(fraction_me)

fraction_lh= ((latino_hispanic)/(total))*100
percentage_lh= round(fraction_lh)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

#Prepare the data for plotting
Black = data[data['Ethnicity']=='Black'].shape[0]
White = data[data['Ethnicity']=='White'].shape[0]
Indian = data[data['Ethnicity']=='Indian'].shape[0]
Latino_Hispanic = data[data['Ethnicity']=='Latino_Hispanic'].shape[0]
Southeast_Asian = data[data['Ethnicity']=='Southeast Asian'].shape[0]
East_Asian = data[data['Ethnicity']=='East Asian'].shape[0]
Middle_Eastern = data[data['Ethnicity']=='Middle Eastern'].shape[0]

#Plot distribution of ethnicities
plt.figure(figsize=(13, 6))
bar= plt.bar(['Black', 'White', 'Indian', 'Latino Hispanic', 'Southeast Asian', 'East Asian', 'Middle Eastern'], [Black, White, Indian, Latino_Hispanic, Southeast_Asian, East_Asian, Middle_Eastern], color=['orange', 'blue', 'green', 'purple', 'red', 'pink', 'olive'])
plt.xlabel('Ethnicity')
plt.ylabel('Number of Entrepreneurs')
plt.title('Distribution of Entrepreneurs Promoted Through DNNL - Ethnicity')
#Add percentages above bars
plt.bar_label(bar, labels=[f'{fraction_black:.1f}%', f'{fraction_white:.1f}%', f'{fraction_indian:.1f}%', f'{fraction_lh:.1f}%', f'{fraction_sa:.1f}%', f'{fraction_ea:.1f}%', f'{fraction_me:.1f}%'], fontsize=12, label_type='edge')

plt.show()